# From MPI to Agentic Workflows with LangGraph

**Author:** Rosa Filgueira  
**Email:** r.filgueira@epcc.ed.ac.uk

This tutorial is designed for readers who understand (or are at least curious about) parallel and distributed computing, but who may never (or very little) have used an AI agent, LangGraph, or LangChain.

We begin with a familiar MPI-style pattern and change one idea at a time. The first six parts deliberately use an extremely small numerical problem:

> Start with the array `[0, 1, 2, ..., 99]`, add `+1` to every value, and reconstruct `[1, 2, 3, ..., 100]`.

Nobody needs an AI agent to add one to an integer. That is exactly why the example is useful. The arithmetic stays trivial, allowing us to focus on:

- who owns the data;
- who creates and receives work;
- how state is represented;
- who decides what happens next;
- how results are checked;
- how failures and ambiguous instructions are handled;
- and where an LLM adds something genuinely new.

In Part 7, the example changes to **IoT sensor triage**. Natural-language diagnostic notes create a real need for interpretation, so an LLM has a sensible and bounded role.

## Learning objectives

By the end of the notebook, you should be able to explain:

1. the difference between an MPI process, an agent, a LangGraph node, and a tool;
2. why an agent does **not** necessarily require an LLM;
3. what LangGraph contributes to a workflow;
4. what LangChain contributes in the final example;
5. how deterministic routing differs from LLM-based interpretation;
6. where agents and actions are defined in the Python code;
7. why explicit validation, state, audit logs, and bounded actions matter.


# The complete journey

The notebook follows this progression:

```text
MPI4py scatter/gather
        ↓
MPI ranks described as manager and workers
        ↓
Plain Python agents requesting work
        ↓
Basic LangGraph workflow
        ↓
Proactive LangGraph validation
        ↓
Deterministic LangGraph task-control loop
        ↓
LLM-powered sensor-triage agent selecting a bounded action
```

Each step preserves ideas from the previous step and adds one new coordination capability.

| Part | Main change | Decision source | LLM? |
|---|---|---|---:|
| **1** | Scatter data, compute locally, gather results | MPI program structure | No |
| **2** | Describe MPI ranks as manager and specialist roles | MPI program structure | No |
| **3** | Workers request tasks from a manager | Explicit Python methods | No |
| **4** | Represent the workflow as state, nodes, and edges | Explicit graph structure | No |
| **5** | Workers inspect and validate tasks | Explicit Python conditions | No |
| **6** | Add loops, splitting, clarification, and recovery | Explicit Python conditions and routing | No |
| **7** | Interpret free-text sensor notes | Deterministic rules **plus** an LLM | Yes |



## When does the notebook become agentic?

Step 2 introduces agent-style names and roles, but the workers still execute fixed MPI instructions. From Step 3 onwards, the notebook is already agentic: the workers have roles, request work, perform actions, validate results, and continue towards a goal.

When LangGraph is introduced in Step 4, some nodes represent those agent behaviours, while other nodes support the workflow by creating tasks, routing work, or merging results. Step 5 adds proactive inspection and validation, Step 6 gives the agents a broader set of deterministic actions and recovery behaviours, and Step 7 adds an LLM only for bounded language interpretation.


# Beginner glossary

| Concept | What it is | In this notebook |
|---|---|---|
| **MPI process** | An independently executing instance of the MPI program, identified by a rank | Ranks split and process the array in Parts 1–2 |
| **LLM** | A model that can interpret and generate language and make a bounded decision from a prompt | Interprets IoT diagnostic notes in Part 7 |
| **Agent** | A component given a role, relevant context or state, and possible actions; it may inspect a situation and decide what to do next | `ManagerAgent`, `PlusOneAgent`, and `SensorTriageAgent` |
| **Action/tool** | An operation that an agent or workflow is allowed to execute | Add one, split a task, accept a reading, retry, request maintenance, or send to human review |
| **LangChain** | A toolkit offering model integrations, prompts, structured output, tools, and higher-level agent interfaces | `ChatOpenAI` connects to the OpenAI model and returns a structured `SensorDecision` in Part 7 |
| **LangGraph** | An orchestration framework for state, nodes, edges, branches, loops, and long-running agentic workflows | Controls task movement, validation, looping, and completion in Parts 4–7 |
| **State** | The data remembered and passed through a workflow | Pending tasks, completed results, logs, current sensor reading, and final output |
| **Node** | A Python function registered as one step in a LangGraph graph | `manager_splitter`, `advanced_plus_one_agent`, `llm_sensor_agent` |
| **Edge** | A connection saying which node runs next | Manager → worker → merger |
| **Conditional edge/router** | Python logic that chooses the next edge based on state | Continue processing while pending tasks remain |
| **Prompt** | Instructions and context sent to an LLM | The SensorTriageAgent policy and one sensor reading |
| **Structured output** | A required response shape rather than unrestricted prose | One permitted `action` plus a short `reason` |
| **Deterministic rule** | An explicit condition whose behaviour is programmed in advance | `battery_percent < 10` → maintenance |
| **Audit log** | A record of what happened and why | `agent_log` and `audit_log` fields |




## Where are the agents defined?

The agents are defined through their roles and behaviour in the notebook code.

- In Part 2, the MPI ranks are given agent-style names, but they still have very little autonomy.
- Part 3 defines `ManagerAgent` and `PlusOneAgent` as Python classes.
- In Part 4, some LangGraph nodes represent the existing manager and worker roles, while other nodes simply prepare tasks or merge results.
- In Part 5, the worker becomes more proactive by inspecting the task and validating its input and output.
- In Part 6, the `advanced_plus_one_agent` node chooses between several deterministic actions, updates the workflow state, and continues towards a goal.
- In Part 7, the `llm_sensor_agent()` node uses an LLM to select a permitted action, while `execute_sensor_action()` carries it out.



## How to use the notebook

For each part:

1. Read the explanation cell first.
2. Run the code cell.
3. Inspect the printed log, not only the final array.
4. Ask: **Where is the decision made?**
5. Ask: **What information was available when it was made?**
6. Ask: **Was that decision programmed explicitly or delegated to an LLM?**

The logs are part of the lesson. They make the coordination visible.


# Setup for Google Colab

Run the next cell once at the beginning of a fresh Colab session.

It installs:

- **Open MPI**, the underlying MPI implementation;
- **mpi4py**, Python bindings for MPI;
- **NumPy**, used for arrays in the MPI examples;
- **LangGraph**, used for stateful graph orchestration;
- **LangChain**, which provides shared model and agent abstractions;
- **langchain-openai**, the integration used to call an OpenAI model in Part 7.

The installation command beginning with `!` is a shell command executed by Colab, not normal Python.

> **Local or HPC environments:** do not automatically run the `apt-get` commands on a managed login node. Use the MPI installation and environment-management process appropriate for your system.


In [ ]:
# Colab setup
# This may take a minute the first time.

!apt-get update -qq
!apt-get install -y -qq openmpi-bin libopenmpi-dev
!pip install -qU mpi4py numpy langgraph langchain langchain-openai


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 80.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.8/247.8 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.6/139.6 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.1/122.1 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.7/561.7 kB 20.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.5.1 which is incompatible.


# Part 1: Basic MPI4py scatter/gather

MPI is a message-passing standard for programs composed of multiple processes. In this Colab example, `mpiexec` starts four copies of the same Python script.

Every process executes the script, but each sees a different:

- **rank**: the process identifier, such as `0`, `1`, `2`, or `3`;
- **size**: the total number of processes.

Rank 0 acts as the **root** process. It initially owns the complete NumPy array.

```text
Rank 0: [0, 1, 2, ..., 99]
             │
             ├── chunk for rank 0
             ├── chunk for rank 1
             ├── chunk for rank 2
             └── chunk for rank 3
```

Each process adds one to its local chunk. The root then reconstructs the final array.

```text
create → scatter → local compute → gather → validate
```

This is a collective, data-parallel pattern. The workers do not decide which operation to perform or what to do if the task is unsuitable.


## Reading the Part 1 code

### The communicator

```python
comm = MPI.COMM_WORLD
```

A communicator is a group of MPI processes that can communicate. `COMM_WORLD` contains every process started for this program.

### Rank and size

```python
rank = comm.Get_rank()
size = comm.Get_size()
```

Every process runs these lines. The values differ by process.

### Counts and displacements

The array length may not divide evenly by the number of ranks.

- `counts[r]` says how many values rank `r` receives.
- `displacements[r]` says where rank `r`'s section begins in the full array.

For 100 values and four ranks:

```text
counts        = [25, 25, 25, 25]
displacements = [ 0, 25, 50, 75]
```

For 101 values, the first rank would receive one extra value.

### `Scatterv`

```python
comm.Scatterv(...)
```

The `v` means that the chunks may have varying lengths. The root distributes the relevant array sections into each process's private `local_data`.

### Local computation

```python
local_data += 1
```

Every rank performs the same operation on its own data.

### `Gatherv`

```python
comm.Gatherv(...)
```

The modified chunks are placed back into the correct positions of the root's result array.

### Validation

Rank 0 independently constructs the expected answer and checks equality. Validation is deterministic and explicit.


In [ ]:
%%writefile mpi_basic_add_one.py
from mpi4py import MPI
import numpy as np

comm = MPI.COMM_WORLD
rank = comm.Get_rank()
size = comm.Get_size()

N = 100

# Rank 0 owns the full array initially.
if rank == 0:
    data = np.arange(N, dtype=np.int32)
    print("Rank 0 created the original array:")
    print(data)
else:
    data = None

# Decide how many values each rank receives.
counts = np.full(size, N // size, dtype=np.int32)
counts[:N % size] += 1

# Displacements say where each rank's chunk starts in the full array.
displacements = np.zeros(size, dtype=np.int32)
displacements[1:] = np.cumsum(counts[:-1])
print("displacements %s" % displacements )

# Each rank allocates space for its local chunk.
local_data = np.empty(counts[rank], dtype=np.int32)

# Scatter chunks from rank 0 to all ranks.
comm.Scatterv(
    [data, counts, displacements, MPI.INT],
    local_data,
    root=0
)

print(f"Rank {rank} received {local_data}")

# Local computation: add +1 to each value.
local_data += 1

print(f"Rank {rank} produced {local_data}")

# Rank 0 allocates space for the final result.
if rank == 0:
    result = np.empty(N, dtype=np.int32)
else:
    result = None

# Gather modified chunks back to rank 0.
comm.Gatherv(
    local_data,
    [result, counts, displacements, MPI.INT],
    root=0
)

if rank == 0:
    print("\nFinal array after adding +1:")
    print(result)

    expected = np.arange(1, N + 1, dtype=np.int32)
    assert np.array_equal(result, expected)
    print("\nValidation passed.")


Writing mpi_basic_add_one.py


In [ ]:
!mpiexec --allow-run-as-root --oversubscribe -n 4 python mpi_basic_add_one.py


Rank 0 created the original array:
[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47
 48 49 50 51 52 53 54 55 56 57 58 59 60 61 62 63 64 65 66 67 68 69 70 71
 72 73 74 75 76 77 78 79 80 81 82 83 84 85 86 87 88 89 90 91 92 93 94 95
 96 97 98 99]
displacements [ 0 25 50 75]
Rank 0 received [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24]
Rank 0 produced [ 1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24
 25]
displacements [ 0 25 50 75]
Rank 3 received [75 76 77 78 79 80 81 82 83 84 85 86 87 88 89 90 91 92 93 94 95 96 97 98
 99]
Rank 3 produced [ 76  77  78  79  80  81  82  83  84  85  86  87  88  89  90  91  92  93
  94  95  96  97  98  99 100]
displacements [ 0 25 50 75]
Rank 2 received [50 51 52 53 54 55 56 57 58 59 60 61 62 63 64 65 66 67 68 69 70 71 72 73
 74]
Rank 2 produced [51 52 53 54 55 56 57 58 59 60 61 62 63 64 65 66 67 68 69 70 71 

## Part 1 explained: what happened at runtime?

With four ranks, the conceptual execution is:

```text
Rank 0 creates 0..99
        ↓
Scatterv distributes four chunks
        ↓
┌────────────┬────────────┬────────────┬────────────┐
│ Rank 0     │ Rank 1     │ Rank 2     │ Rank 3     │
│ 0..24      │ 25..49     │ 50..74     │ 75..99     │
│ add +1     │ add +1     │ add +1     │ add +1     │
└────────────┴────────────┴────────────┴────────────┘
        ↓
Gatherv reconstructs 1..100 on Rank 0
        ↓
Rank 0 validates the result
```

The order of printed lines can vary because the processes execute independently. Seeing Rank 2 print before Rank 1 does not mean the result is wrong.

## Where is the decision?

There is almost no runtime choice:

- rank 0 is chosen by the programmer as the root;
- the array distribution is calculated by explicit code;
- every rank always performs `+1`;
- the program always gathers and validates.

MPI executes a predetermined parallel computation.


## Part 1 takeaways

| Question | Answer |
|---|---|
| Who owns the original data? | Rank 0 |
| Who decides the operation? | The programmer |
| Do workers inspect the task? | No |
| Can a worker refuse or modify a task? | No |
| Is there shared Python memory? | No; each process has its own memory |
| Is an LLM used? | No |
| Is this agentic? | Not in the sense used later; it is a fixed parallel program |

The strength of this model is precision and performance. When the computation is well defined, passive workers are often exactly what we want.


# Part 2: MPI ranks described as manager and specialist agents

The computation is still MPI, but the roles are now named:

```text
rank 0  → ArrayManagerAgent
rank 1+ → PlusOneAgent
```

The manager creates task dictionaries and sends them to specialist workers. Each task includes:

```python
{
    "agent_role": "PlusOneAgent",
    "operation": "+1",
    "start_position": ...,
    "chunk": ...
}
```

This makes the communication more semantic: the message describes both the data and the requested operation.

However, renaming a process does not automatically make it autonomous.

```text
manager sends fixed task
        ↓
worker receives it
        ↓
worker performs exactly +1
        ↓
worker returns result
```


## Messages and tags in Part 2

MPI tags allow the receiver to distinguish message types:

| Tag | Meaning |
|---|---|
| `TAG_TASK` | A task is being sent |
| `TAG_RESULT` | A completed result is being returned |
| `TAG_STOP` | No work should be performed |

The manager uses point-to-point communication:

```python
comm.send(task, dest=worker_rank, tag=TAG_TASK)
```

A worker receives from rank 0 and inspects the message tag:

```python
task = comm.recv(source=0, tag=MPI.ANY_TAG, status=status)
```

Results may arrive in any order:

```python
response = comm.recv(source=MPI.ANY_SOURCE, tag=TAG_RESULT)
```

The `start_position` field is therefore essential. It lets the manager place each returned chunk in the correct location regardless of arrival order.

## Where is the “agent” defined?

`plus_one_agent(chunk)` is an ordinary Python function playing a specialist worker role. In Part 2, it still follows a fixed instruction, so this is an agent-style role rather than the more agentic behaviour introduced in Part 3.


In [ ]:
%%writefile mpi_agentic_add_one.py
from mpi4py import MPI
import numpy as np

comm = MPI.COMM_WORLD
rank = comm.Get_rank()
size = comm.Get_size()

TAG_TASK = 1
TAG_RESULT = 2
TAG_STOP = 3

N = 100


def plus_one_agent(chunk):
    """
    Specialist PlusOneAgent:
    receives part of the array and adds +1 to every position.
    """
    return chunk + 1


if rank == 0:
    # -------------------------------
    # ArrayManagerAgent
    # -------------------------------
    data = np.arange(N, dtype=np.int32)

    print("ArrayManagerAgent has the original array:")
    print(data)

    if size == 1:
        result = data + 1

    else:
        number_of_workers = size - 1
        chunks = np.array_split(data, number_of_workers)
        print("chunkcs is %s" %chunks)

        result = np.empty_like(data)

        start_position = 0
        active_workers = 0

        # Send work to PlusOneAgents.
        for worker_rank, chunk in enumerate(chunks, start=1):
            if len(chunk) == 0:
                comm.send(None, dest=worker_rank, tag=TAG_STOP)
                continue

            task = {
                "agent_role": "PlusOneAgent",
                "operation": "+1",
                "start_position": start_position,
                "chunk": chunk,
            }

            comm.send(task, dest=worker_rank, tag=TAG_TASK)

            print(
                f"ArrayManagerAgent sent positions "
                f"{start_position} to {start_position + len(chunk) - 1} "
                f"to PlusOneAgent {worker_rank}"
            )

            start_position += len(chunk)
            active_workers += 1

        # Receive results back from PlusOneAgents.
        for _ in range(active_workers):
            response = comm.recv(source=MPI.ANY_SOURCE, tag=TAG_RESULT)

            worker_rank = response["worker_rank"]
            start = response["start_position"]
            modified_chunk = response["modified_chunk"]

            result[start:start + len(modified_chunk)] = modified_chunk

            print(
                f"ArrayManagerAgent received modified positions "
                f"{start} to {start + len(modified_chunk) - 1} "
                f"from PlusOneAgent {worker_rank}"
            )

    print("\nFinal array after all PlusOneAgents finished:")
    print(result)

    expected = np.arange(1, N + 1, dtype=np.int32)
    assert np.array_equal(result, expected)
    print("\nValidation passed.")


else:
    # -------------------------------
    # PlusOneAgent
    # -------------------------------
    status = MPI.Status()
    task = comm.recv(source=0, tag=MPI.ANY_TAG, status=status)

    if status.Get_tag() == TAG_STOP:
        pass

    else:
        chunk = task["chunk"]
        start_position = task["start_position"]

        print(
            f"PlusOneAgent {rank} received positions "
            f"{start_position} to {start_position + len(chunk) - 1}"
        )

        modified_chunk = plus_one_agent(chunk)

        response = {
            "worker_rank": rank,
            "start_position": start_position,
            "modified_chunk": modified_chunk,
        }

        comm.send(response, dest=0, tag=TAG_RESULT)


Writing mpi_agentic_add_one.py


In [ ]:
!mpiexec --allow-run-as-root --oversubscribe -n 4 python mpi_agentic_add_one.py


ArrayManagerAgent has the original array:
[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47
 48 49 50 51 52 53 54 55 56 57 58 59 60 61 62 63 64 65 66 67 68 69 70 71
 72 73 74 75 76 77 78 79 80 81 82 83 84 85 86 87 88 89 90 91 92 93 94 95
 96 97 98 99]
chunkcs is [array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16,
       17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33],
      dtype=int32), array([34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50,
       51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66],
      dtype=int32), array([67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83,
       84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99],
      dtype=int32)]
PlusOneAgent 2 received positions 34 to 66
ArrayManagerAgent sent positions 0 to 33 to PlusOneAgent 1
PlusOneAgent 1 received positions

## Part 2 explained: what changed and what did not?

| Aspect | Part 1 | Part 2 |
|---|---|---|
| Communication | MPI collectives | MPI point-to-point messages |
| Root role | Rank 0 | `ArrayManagerAgent` |
| Worker role | Rank | `PlusOneAgent` |
| Work description | Implicit in program | Explicit task dictionary |
| Result placement | `Gatherv` metadata | `start_position` in response |
| Worker autonomy | None | Still almost none |
| LLM | No | No |

The vocabulary is useful because it reveals a reusable coordination structure:

```text
manager owns problem state
workers have specialist capabilities
messages contain tasks and results
manager reconstructs the whole answer
```

But the workers remain passive. They wait for one message, execute one fixed function, and stop.


## Part 2 takeaways

Calling something an agent is a **design description**, not a technical guarantee.

A component becomes more recognisably agentic when it can do some combination of the following:

- request work when ready;
- inspect instructions;
- select among actions;
- validate an outcome;
- retain relevant state;
- react to failure;
- ask for clarification;
- continue until a goal or stopping condition is reached.

Part 2 provides the manager/specialist vocabulary. Part 3 adds a task queue and pull-based work.


# Part 3: Plain Python agents requesting work

MPI is removed so that we can focus entirely on coordination.

The program now contains:

- a `ManagerAgent` that owns a task queue and the global result;
- several `PlusOneAgent` objects;
- `Task` objects describing work;
- `Result` objects describing completed work.

The key behavioural change is from **push** to **pull**:

```text
Push model
Manager → “Here is your fixed task.”

Pull model
Worker  → “I am ready. Do you have a task?”
Manager → “Yes. Take the next one.”
```

This is the first **recognisably agentic step**. The important difference is not the arithmetic; it is the interaction pattern. The worker participates in the workflow. It requests a task, processes it, validates its own result, reports back, and asks whether more work remains. The agents are still deterministic Python objects. They do not contain an LLM. Their capabilities are methods written by the programmer.




## Reading the Part 3 code

### `Task`

A task records:

- a unique `task_id`;
- where the chunk belongs in the original array;
- the values to process.

### `Result`

A result records:

- which agent produced it;
- which task it belongs to;
- where it should be inserted;
- the modified values.

### `ManagerAgent`

The manager has both **data state** and **control state**:

| Field | Purpose |
|---|---|
| `original_array` | Source data |
| `chunk_size` | Programmer-supplied partition size |
| `pending_tasks` | Work not yet completed |
| `results` | Returned task results |
| `final_array` | Reconstructed global result |

Its methods create work, answer requests, receive results, and validate completion.

### `PlusOneAgent`

Each worker has:

- a name;
- a reference to the manager;
- a loop for requesting work;
- a specialist processing method;
- self-validation logic.

The actual action remains ordinary Python:

```python
modified_values = [x + 1 for x in task.values]
```


In [ ]:
from dataclasses import dataclass
from typing import Optional


@dataclass
class Task:
    task_id: int
    start_position: int
    values: list[int]


@dataclass
class Result:
    agent_name: str
    task_id: int
    start_position: int
    modified_values: list[int]


class ManagerAgent:
    """
    The manager owns the full array, creates tasks, and rebuilds the final array.
    """

    def __init__(self, array: list[int], chunk_size: int = 20):
        self.original_array = array
        self.chunk_size = chunk_size
        self.pending_tasks: list[Task] = []
        self.results: list[Result] = []
        self.final_array: list[Optional[int]] = [None] * len(array)

    def create_tasks(self):
        task_id = 0

        for start in range(0, len(self.original_array), self.chunk_size):
            chunk = self.original_array[start:start + self.chunk_size]

            self.pending_tasks.append(
                Task(
                    task_id=task_id,
                    start_position=start,
                    values=chunk,
                )
            )

            task_id += 1

        print(f"ManagerAgent created {len(self.pending_tasks)} tasks.")

    def request_task(self, agent_name: str) -> Optional[Task]:
        """
        Agents call this when they are ready.
        """
        if not self.pending_tasks:
            print(f"{agent_name} asked for work, but no tasks remain.")
            return None

        task = self.pending_tasks.pop(0)

        print(
            f"{agent_name} proactively requested work "
            f"and received task {task.task_id}."
        )

        return task

    def receive_result(self, result: Result):
        self.results.append(result)

        start = result.start_position
        end = start + len(result.modified_values)

        self.final_array[start:end] = result.modified_values

        print(
            f"ManagerAgent received result for task {result.task_id} "
            f"from {result.agent_name}."
        )

    def validate_final_array(self):
        expected = [x + 1 for x in self.original_array]

        if self.final_array != expected:
            raise RuntimeError("Final array is incorrect.")

        print("ManagerAgent validated the final array.")


class PlusOneAgent:
    """
    Specialist agent that can add +1.

    This agent is proactive because it asks for work and validates itself.
    """

    def __init__(self, name: str, manager: ManagerAgent):
        self.name = name
        self.manager = manager

    def run_until_no_work_left(self):
        while True:
            task = self.manager.request_task(self.name)

            if task is None:
                break

            result = self.process_task(task)
            self.manager.receive_result(result)

    def process_task(self, task: Task) -> Result:
        print(
            f"{self.name} is processing positions "
            f"{task.start_position} to "
            f"{task.start_position + len(task.values) - 1}."
        )

        modified_values = [x + 1 for x in task.values]

        # Self-validation
        for original, modified in zip(task.values, modified_values):
            if modified != original + 1:
                raise RuntimeError(
                    f"{self.name} detected an error in task {task.task_id}."
                )

        print(f"{self.name} validated task {task.task_id}.")

        return Result(
            agent_name=self.name,
            task_id=task.task_id,
            start_position=task.start_position,
            modified_values=modified_values,
        )


# Run the simple agentic example.
array = list(range(100))

manager = ManagerAgent(array, chunk_size=20)
manager.create_tasks()

agents = [
    PlusOneAgent("PlusOneAgent-1", manager),
    PlusOneAgent("PlusOneAgent-2", manager),
    PlusOneAgent("PlusOneAgent-3", manager),
    PlusOneAgent("PlusOneAgent-4", manager),
]

# Each agent keeps asking for work.
for agent in agents:
    agent.run_until_no_work_left()

manager.validate_final_array()

print("\nOriginal array:")
print(manager.original_array)

print("\nFinal array:")
print(manager.final_array)


ManagerAgent created 5 tasks.
PlusOneAgent-1 proactively requested work and received task 0.
PlusOneAgent-1 is processing positions 0 to 19.
PlusOneAgent-1 validated task 0.
ManagerAgent received result for task 0 from PlusOneAgent-1.
PlusOneAgent-1 proactively requested work and received task 1.
PlusOneAgent-1 is processing positions 20 to 39.
PlusOneAgent-1 validated task 1.
ManagerAgent received result for task 1 from PlusOneAgent-1.
PlusOneAgent-1 proactively requested work and received task 2.
PlusOneAgent-1 is processing positions 40 to 59.
PlusOneAgent-1 validated task 2.
ManagerAgent received result for task 2 from PlusOneAgent-1.
PlusOneAgent-1 proactively requested work and received task 3.
PlusOneAgent-1 is processing positions 60 to 79.
PlusOneAgent-1 validated task 3.
ManagerAgent received result for task 3 from PlusOneAgent-1.
PlusOneAgent-1 proactively requested work and received task 4.
PlusOneAgent-1 is processing positions 80 to 99.
PlusOneAgent-1 validated task 4.
Ma

## Part 3 explained: one task's lifecycle

```text
ManagerAgent.create_tasks()
        ↓
pending_tasks contains five Task objects
        ↓
PlusOneAgent asks request_task()
        ↓
Manager removes and returns the first task
        ↓
PlusOneAgent.process_task()
        ↓
worker computes and self-validates
        ↓
ManagerAgent.receive_result()
        ↓
result is stored in its original position
        ↓
worker requests another task
```

There are two levels of validation:

1. **Local self-validation** checks each processed pair.
2. **Global validation** checks the complete reconstructed array.

This layered validation is a useful workflow pattern even when the real computation is far more complex than `+1`.


## Important simplification: these agents are not running concurrently

The example creates four agent objects, but then calls:

```python
for agent in agents:
    agent.run_until_no_work_left()
```

The first agent is allowed to keep requesting work until the queue is empty. Therefore, this cell illustrates the **interaction protocol**, not fair scheduling or parallel execution.

A production implementation might use:

- threads;
- processes;
- asynchronous tasks;
- distributed workers;
- an external queue;
- or LangGraph branches.

Keeping Part 3 sequential makes the manager/worker conversation easy to inspect.

## Is this really an agent?

It is a small deterministic software agent:

- it has a role;
- it can perceive task data;
- it can act;
- it validates its action;
- it repeatedly requests work;
- it stops when its goal condition is reached.

It is not an AI language agent, and it has no open-ended reasoning.


## Part 3 takeaways

| Question | Answer |
|---|---|
| Who creates tasks? | `ManagerAgent` |
| Who requests tasks? | `PlusOneAgent` |
| Where is state stored? | Inside Python objects |
| How is the next action selected? | Fixed method logic |
| Who validates? | Worker locally and manager globally |
| Is a graph framework used? | No |
| Is an LLM used? | No |

The control loop is still handwritten. Part 4 expresses a similar workflow explicitly as a graph.


# Part 4: A basic LangGraph workflow

LangGraph represents an application as a graph operating on shared state.

The basic vocabulary is:

```text
State  = what the workflow remembers
Node   = a Python function that reads and updates state
Edge   = which node runs next
Router = a Python function that chooses among edges
```

For this part:

```text
START
  ↓
manager_splitter
  ↓
fan out one branch per task
  ↓
plus_one_agent
  ↓
fan in collected results
  ↓
manager_merger
  ↓
END
```

LangGraph does not require an LLM. Every node in this part is deterministic Python.


## Understanding LangGraph state

`SimpleGraphState` is a `TypedDict`. It documents the fields that may travel through the graph.

| State field | Meaning |
|---|---|
| `array` | Original input |
| `chunk_size` | Requested partition size |
| `tasks` | Work produced by the manager |
| `results` | Results returned by worker branches |
| `agent_log` | Human-readable audit messages |
| `final_array` | Reconstructed answer |

Two fields use reducers:

```python
results: Annotated[list[WorkerResult], operator.add]
agent_log: Annotated[list[str], operator.add]
```

When multiple worker branches return lists, LangGraph needs to know how to combine them. `operator.add` means concatenate the lists rather than letting one branch overwrite another.

This is especially important in fan-out/fan-in patterns.


## Understanding nodes, `Send`, edges, compilation, and invocation

### Nodes

```python
graph_builder.add_node("manager_splitter", manager_splitter)
```

This registers a normal Python function under a graph name.

### Dynamic fan-out with `Send`

`dispatch_to_workers()` returns one `Send` object for every task:

```python
Send("plus_one_agent", {"task": task})
```

Conceptually:

```text
one manager state
      ↓
 task 0 → plus_one_agent
 task 1 → plus_one_agent
 task 2 → plus_one_agent
 task 3 → plus_one_agent
 task 4 → plus_one_agent
      ↓
merged result lists
```

The same node function handles every branch.

### Edges

Edges define legal transitions. They do not perform reasoning.

### Compile

```python
simple_langgraph = graph_builder.compile()
```

Compilation turns the graph definition into an executable application.

### Invoke

```python
result = simple_langgraph.invoke(initial_state)
```

Invocation supplies the starting state and runs until `END`.


In [ ]:
from typing import Annotated, TypedDict
import operator

from langgraph.graph import StateGraph, START, END
from langgraph.types import Send


class WorkerTask(TypedDict):
    worker_id: int
    start: int
    chunk: list[int]


class WorkerResult(TypedDict):
    worker_id: int
    start: int
    modified_chunk: list[int]


class SimpleGraphState(TypedDict):
    array: list[int]
    chunk_size: int
    tasks: list[WorkerTask]
    results: Annotated[list[WorkerResult], operator.add]
    agent_log: Annotated[list[str], operator.add]
    final_array: list[int]


def manager_splitter(state: SimpleGraphState) -> dict:
    """
    ManagerAgent:
    creates tasks from the original array.
    """
    array = state["array"]
    chunk_size = state["chunk_size"]

    tasks = []
    worker_id = 0

    for start in range(0, len(array), chunk_size):
        chunk = array[start:start + chunk_size]

        tasks.append({
            "worker_id": worker_id,
            "start": start,
            "chunk": chunk,
        })

        worker_id += 1

    return {
        "tasks": tasks,
        "results": [],
        "agent_log": [
            f"ManagerAgent created {len(tasks)} tasks."
        ],
    }


def dispatch_to_workers(state: SimpleGraphState):
    """
    Dynamic routing:
    send each task to the same PlusOneAgent node.

    LangGraph will run one branch per task and then merge the results.
    """
    return [
        Send("plus_one_agent", {"task": task})
        for task in state["tasks"]
    ]


def plus_one_agent(worker_state: dict) -> dict:
    """
    PlusOneAgent:
    receives one task and adds +1 to its chunk.
    """
    task = worker_state["task"]

    worker_id = task["worker_id"]
    start = task["start"]
    chunk = task["chunk"]

    modified_chunk = [x + 1 for x in chunk]

    return {
        "results": [{
            "worker_id": worker_id,
            "start": start,
            "modified_chunk": modified_chunk,
        }],
        "agent_log": [
            f"PlusOneAgent-{worker_id} processed positions "
            f"{start} to {start + len(chunk) - 1}."
        ],
    }


def manager_merger(state: SimpleGraphState) -> dict:
    """
    ManagerAgent:
    reconstructs the final array.
    """
    final_array = [None] * len(state["array"])

    for result in state["results"]:
        start = result["start"]
        chunk = result["modified_chunk"]
        final_array[start:start + len(chunk)] = chunk

    expected = [x + 1 for x in state["array"]]
    assert final_array == expected

    return {
        "final_array": final_array,
        "agent_log": [
            "ManagerAgent merged all results and validated the final array."
        ],
    }


# Build the graph.
graph_builder = StateGraph(SimpleGraphState)

graph_builder.add_node("manager_splitter", manager_splitter)
graph_builder.add_node("plus_one_agent", plus_one_agent)
graph_builder.add_node("manager_merger", manager_merger)

graph_builder.add_edge(START, "manager_splitter")

graph_builder.add_conditional_edges(
    "manager_splitter",
    dispatch_to_workers,
    ["plus_one_agent"],
)

graph_builder.add_edge("plus_one_agent", "manager_merger")
graph_builder.add_edge("manager_merger", END)

simple_langgraph = graph_builder.compile()


# Run the graph.
initial_array = list(range(100))

result = simple_langgraph.invoke({
    "array": initial_array,
    "chunk_size": 20,
    "tasks": [],
    "results": [],
    "agent_log": [],
    "final_array": [],
})

print("Agent log:")
for line in result["agent_log"]:
    print("-", line)

print("\nFinal array:")
print(result["final_array"])


Agent log:
- ManagerAgent created 5 tasks.
- PlusOneAgent-0 processed positions 0 to 19.
- PlusOneAgent-1 processed positions 20 to 39.
- PlusOneAgent-2 processed positions 40 to 59.
- PlusOneAgent-3 processed positions 60 to 79.
- PlusOneAgent-4 processed positions 80 to 99.
- ManagerAgent merged all results and validated the final array.

Final array:
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100]


## Part 4 explained: what LangGraph added

The arithmetic did not change. The orchestration became explicit.

```text
manager_splitter:
    array → task descriptions

dispatch_to_workers:
    task descriptions → worker branches

plus_one_agent:
    one task → one result + one log entry

reducers:
    many branch outputs → combined lists

manager_merger:
    combined results → validated final array
```

This separation helps when workflows grow:

- each node has a clear responsibility;
- state fields are visible;
- branches and loops are represented explicitly;
- routing can be tested separately;
- logging and checkpoints can be added consistently.

## Node versus agent

`plus_one_agent` is both:

- technically, a **LangGraph node function**;
- conceptually, a **specialist agent role**.

`manager_splitter` is also a node, but it behaves more like a deterministic preparation step than an autonomous agent. Not every node needs to be called an agent.


## Part 4 takeaways

| Question | Answer |
|---|---|
| What does LangGraph remember? | The fields in `SimpleGraphState` |
| What performs computation? | Python node functions |
| What creates worker branches? | `dispatch_to_workers()` returning `Send` objects |
| How are branch outputs combined? | Reducers using `operator.add` |
| How does LangGraph know the route? | The programmer-defined edges and dispatcher |
| Does LangGraph decide the operation? | No |
| Is an LLM used? | No |

Part 5 keeps the graph structure but makes the worker inspect the task before acting.


# Part 5: Proactive inspection and validation

The worker now examines the task around the arithmetic.

It asks:

```text
Is the requested operation supported?
        ↓ yes
Are all input values valid integers?
        ↓ yes
Perform +1
        ↓
Does every output equal input + 1?
        ↓ yes
Return a validated result
```

If a check fails, the node raises an exception rather than silently returning an unsafe result.

This is more proactive because the worker does not blindly execute whatever it receives. It applies a capability check, input-quality check, and output-validation check.

All decisions are still explicit Python conditions.


## The deterministic decision tree in Part 5

```text
Receive task
   │
   ├── operation != "+1"
   │       └── raise ValueError
   │
   ├── any value is not an integer
   │       └── raise ValueError
   │
   ├── compute modified chunk
   │
   ├── self-validation fails
   │       └── raise RuntimeError
   │
   └── return result and log
```

The worker does not “understand” the task in a language-model sense. It compares structured values against conditions written by the programmer.

For example:

```python
if operation != "+1":
```

This rule has exactly two outcomes. The same input produces the same branch every time.


## New information in the state

Each `ProactiveTask` now includes:

```python
operation: str
```

This matters because the worker receives not only data but also an explicit instruction.

The agent log records intermediate checks:

```text
inspected task
accepted operation
checked integer inputs
completed and validated output
```

An audit log is valuable because a correct final answer does not explain how it was obtained. In larger scientific workflows, logs can record:

- parameters;
- tool versions;
- validation outcomes;
- retry attempts;
- provenance;
- and reasons for escalation.


In [ ]:
from typing import Annotated, TypedDict
import operator

from langgraph.graph import StateGraph, START, END
from langgraph.types import Send


class ProactiveTask(TypedDict):
    worker_id: int
    start: int
    chunk: list[int]
    operation: str


class ProactiveResult(TypedDict):
    worker_id: int
    start: int
    modified_chunk: list[int]


class ProactiveState(TypedDict):
    array: list[int]
    chunk_size: int
    tasks: list[ProactiveTask]
    results: Annotated[list[ProactiveResult], operator.add]
    agent_log: Annotated[list[str], operator.add]
    final_array: list[int]


def proactive_manager_splitter(state: ProactiveState) -> dict:
    array = state["array"]
    chunk_size = state["chunk_size"]

    tasks = []
    worker_id = 0

    for start in range(0, len(array), chunk_size):
        chunk = array[start:start + chunk_size]

        tasks.append({
            "worker_id": worker_id,
            "start": start,
            "chunk": chunk,
            "operation": "+1",
        })

        worker_id += 1

    return {
        "tasks": tasks,
        "results": [],
        "agent_log": [
            f"ManagerAgent created {len(tasks)} tasks with operation '+1'."
        ],
    }


def proactive_dispatch(state: ProactiveState):
    return [
        Send("proactive_plus_one_agent", {"task": task})
        for task in state["tasks"]
    ]


def proactive_plus_one_agent(worker_state: dict) -> dict:
    task = worker_state["task"]

    worker_id = task["worker_id"]
    start = task["start"]
    chunk = task["chunk"]
    operation = task["operation"]

    log = []
    log.append(f"PlusOneAgent-{worker_id} inspected its task.")

    # Check whether this agent can perform the requested operation.
    if operation != "+1":
        raise ValueError(
            f"PlusOneAgent-{worker_id} cannot perform operation {operation}."
        )

    log.append(f"PlusOneAgent-{worker_id} accepted operation '+1'.")

    # Check input quality.
    if not all(isinstance(x, int) for x in chunk):
        raise ValueError(
            f"PlusOneAgent-{worker_id} found non-integer values."
        )

    log.append(f"PlusOneAgent-{worker_id} checked that all values are integers.")

    # Perform the work.
    modified_chunk = [x + 1 for x in chunk]

    # Self-validation.
    for original, modified in zip(chunk, modified_chunk):
        if modified != original + 1:
            raise RuntimeError(
                f"PlusOneAgent-{worker_id} failed self-validation."
            )

    log.append(
        f"PlusOneAgent-{worker_id} completed and validated positions "
        f"{start} to {start + len(chunk) - 1}."
    )

    return {
        "results": [{
            "worker_id": worker_id,
            "start": start,
            "modified_chunk": modified_chunk,
        }],
        "agent_log": log,
    }


def proactive_manager_merger(state: ProactiveState) -> dict:
    final_array = [None] * len(state["array"])

    for result in state["results"]:
        start = result["start"]
        chunk = result["modified_chunk"]
        final_array[start:start + len(chunk)] = chunk

    expected = [x + 1 for x in state["array"]]
    assert final_array == expected

    return {
        "final_array": final_array,
        "agent_log": [
            "ManagerAgent merged and validated the final array."
        ],
    }


graph_builder = StateGraph(ProactiveState)

graph_builder.add_node("proactive_manager_splitter", proactive_manager_splitter)
graph_builder.add_node("proactive_plus_one_agent", proactive_plus_one_agent)
graph_builder.add_node("proactive_manager_merger", proactive_manager_merger)

graph_builder.add_edge(START, "proactive_manager_splitter")

graph_builder.add_conditional_edges(
    "proactive_manager_splitter",
    proactive_dispatch,
    ["proactive_plus_one_agent"],
)

graph_builder.add_edge("proactive_plus_one_agent", "proactive_manager_merger")
graph_builder.add_edge("proactive_manager_merger", END)

proactive_langgraph = graph_builder.compile()


initial_array = list(range(100))

result = proactive_langgraph.invoke({
    "array": initial_array,
    "chunk_size": 20,
    "tasks": [],
    "results": [],
    "agent_log": [],
    "final_array": [],
})

print("Agent log:")
for line in result["agent_log"]:
    print("-", line)

print("\nFinal array:")
print(result["final_array"])


Agent log:
- ManagerAgent created 5 tasks with operation '+1'.
- PlusOneAgent-0 inspected its task.
- PlusOneAgent-0 accepted operation '+1'.
- PlusOneAgent-0 checked that all values are integers.
- PlusOneAgent-0 completed and validated positions 0 to 19.
- PlusOneAgent-1 inspected its task.
- PlusOneAgent-1 accepted operation '+1'.
- PlusOneAgent-1 checked that all values are integers.
- PlusOneAgent-1 completed and validated positions 20 to 39.
- PlusOneAgent-2 inspected its task.
- PlusOneAgent-2 accepted operation '+1'.
- PlusOneAgent-2 checked that all values are integers.
- PlusOneAgent-2 completed and validated positions 40 to 59.
- PlusOneAgent-3 inspected its task.
- PlusOneAgent-3 accepted operation '+1'.
- PlusOneAgent-3 checked that all values are integers.
- PlusOneAgent-3 completed and validated positions 60 to 79.
- PlusOneAgent-4 inspected its task.
- PlusOneAgent-4 accepted operation '+1'.
- PlusOneAgent-4 checked that all values are integers.
- PlusOneAgent-4 complet

## Part 5 explained: proactive but not adaptive

The worker has gained **inspection** and **validation**, but the wider graph still follows one fixed structure:

```text
split → process → merge
```

If a task is unsuitable, the workflow stops with an exception.

It cannot yet:

- correct an ambiguous operation;
- split an oversized task;
- requeue work after a simulated failure;
- let another agent take over;
- continue processing the remaining queue;
- report bad data in shared state.

Those behaviours require a loop and a richer state model, introduced in Part 6.


## Part 5 takeaways

| Capability | Present? |
|---|---:|
| Inspect operation | Yes |
| Check input types | Yes |
| Perform computation | Yes |
| Self-validate output | Yes |
| Maintain an audit log | Yes |
| Recover from a failed task | No |
| Split work dynamically | No |
| Ask for clarification | No |
| Use an LLM | No |

“Proactive” here means that the node performs checks around its assigned computation. It does not mean that it has unrestricted autonomy.


# Part 6: Deterministic LangGraph task-control loop

Part 6 adds adaptation while remaining fully deterministic.

The graph contains a queue and repeatedly runs one worker node until the queue is empty:

```text
START
  ↓
create initial task
  ↓
worker takes next task
  ↓
apply programmed decision rules
  ↓
update queue/results/log
  ↓
pending tasks?
  ├── yes → worker again
  └── no  → merge and validate → END
```

The worker can:

- ask for clarification when the operation is wrong;
- split a chunk larger than a configured threshold;
- report bad values;
- simulate failure and requeue a task;
- process and self-validate a valid task;
- request more work.

**LangGraph does not invent any of these behaviours.** Each branch is written in `advanced_plus_one_agent()`.


## The Part 6 state, field by field

| State field | What it stores |
|---|---|
| `original_array` | The full source array |
| `pending_tasks` | Tasks waiting to be handled |
| `completed_results` | Successfully processed chunks |
| `agent_log` | A chronological explanation of actions |
| `bad_task_reports` | Problems that prevent safe completion |
| `next_task_id` | Identifier for newly created split tasks |
| `max_chunk_size` | Programmer-defined “too large” threshold |
| `worker_names` | Names used to simulate multiple specialists |
| `worker_cursor` | Selects the next worker name |
| `final_array` | Reconstructed output |
| `simulate_failure_task_ids` | Tutorial configuration for forced failures |
| `initial_operation` | Initial operation, which may deliberately be wrong |

The state acts like the workflow's shared working memory.

A node receives the current state and returns updates. LangGraph combines those updates into the state used by the next node.


## How the worker chooses an action

The order of checks matters:

```text
Take one task from pending_tasks
        ↓
1. Is operation different from "+1"?
        └── clarify and requeue
        ↓
2. Is len(chunk) greater than max_chunk_size?
        └── split into two new tasks
        ↓
3. Does the chunk contain a non-integer?
        └── record a bad-task report
        ↓
4. Is this task configured to fail on its first attempt?
        └── increment attempts and requeue for takeover
        ↓
5. Otherwise:
        compute +1, self-validate, store result
```

This is a decision tree, but it is not LLM reasoning. The programmer anticipated every branch.

## How large is “too large”?

The demonstration calls:

```python
max_chunk_size=25
```

The code tests:

```python
if len(chunk) > state["max_chunk_size"]:
```

Therefore:

```text
100 values → too large → split into 50 + 50
50 values  → too large → split into 25 + 25
25 values  → accepted and processed
```

The number `25` is an educational configuration value. LangGraph does not derive it. A real system might derive a threshold from memory, runtime estimates, GPU limits, scheduler policy, or historical measurements.


## How LangGraph knows which node runs next

The router `advanced_should_continue()` looks only at workflow state:

```python
if len(state["pending_tasks"]) > 0:
    return "advanced_plus_one_agent"

return "advanced_manager_merge_results"
```

LangGraph then follows the matching conditional edge.

```text
pending_tasks is not empty
        ↓
advanced_plus_one_agent

pending_tasks is empty
        ↓
advanced_manager_merge_results
```

This distinction is crucial:

- **The router function decides**, according to Python code.
- **LangGraph executes the returned route**.
- **No LLM is involved**.

The graph is adaptive in the sense that different state produces different paths, but every path is programmed in advance.


In [ ]:
from typing import TypedDict, Optional
from langgraph.graph import StateGraph, START, END


class AdvancedTask(TypedDict):
    task_id: int
    start: int
    chunk: list
    operation: str
    attempts: int
    assigned_agent: Optional[str]


class AdvancedResult(TypedDict):
    task_id: int
    start: int
    modified_chunk: list[int]
    agent_name: str


class AdvancedState(TypedDict):
    original_array: list
    pending_tasks: list[AdvancedTask]
    completed_results: list[AdvancedResult]
    agent_log: list[str]
    bad_task_reports: list[str]
    next_task_id: int
    max_chunk_size: int
    worker_names: list[str]
    worker_cursor: int
    final_array: list
    simulate_failure_task_ids: list[int]
    initial_operation: str


def advanced_manager_create_initial_tasks(state: AdvancedState) -> dict:
    """
    ManagerAgent owns the original array and creates the initial task.

    We deliberately create one large task so that a PlusOneAgent can decide
    to split it.
    """

    initial_task: AdvancedTask = {
        "task_id": 0,
        "start": 0,
        "chunk": state["original_array"],
        "operation": state["initial_operation"],
        "attempts": 0,
        "assigned_agent": None,
    }

    return {
        "pending_tasks": [initial_task],
        "completed_results": [],
        "agent_log": [
            "ManagerAgent created one initial task containing the full array."
        ],
        "bad_task_reports": [],
        "next_task_id": 1,
        "worker_cursor": 0,
        "final_array": [],
    }


def advanced_should_continue(state: AdvancedState) -> str:
    """
    Router:
    if there is pending work, activate a PlusOneAgent.
    otherwise, merge the final result.
    """

    if len(state["pending_tasks"]) > 0:
        return "advanced_plus_one_agent"

    return "advanced_manager_merge_results"


def advanced_plus_one_agent(state: AdvancedState) -> dict:
    """
    Proactive PlusOneAgent.

    It can:
    - ask for / take work from the queue
    - ask for clarification if the operation is wrong
    - split a chunk if it is too large
    - detect bad values and report them
    - simulate failure and let another agent take over
    - process and validate the task
    - request more work by returning to the graph loop
    """

    pending_tasks = list(state["pending_tasks"])
    completed_results = list(state["completed_results"])
    agent_log = list(state["agent_log"])
    bad_task_reports = list(state["bad_task_reports"])

    task = pending_tasks.pop(0)

    worker_names = state["worker_names"]
    worker_cursor = state["worker_cursor"]

    agent_name = task["assigned_agent"] or worker_names[worker_cursor % len(worker_names)]
    worker_cursor += 1

    task_id = task["task_id"]
    chunk = task["chunk"]
    start = task["start"]
    operation = task["operation"]
    attempts = task["attempts"]

    agent_log.append(
        f"{agent_name} proactively requested work and received task {task_id}."
    )

    # 1. If the manager gives the wrong operation, ask for clarification.
    if operation != "+1":
        agent_log.append(
            f"{agent_name} received unsupported operation '{operation}' "
            f"for task {task_id}. It asked ManagerAgent for clarification."
        )

        clarified_task = dict(task)
        clarified_task["operation"] = "+1"
        clarified_task["attempts"] += 1

        pending_tasks.insert(0, clarified_task)

        agent_log.append(
            f"ManagerAgent clarified task {task_id}: the correct operation is '+1'."
        )

        return {
            "pending_tasks": pending_tasks,
            "completed_results": completed_results,
            "agent_log": agent_log,
            "bad_task_reports": bad_task_reports,
            "next_task_id": state["next_task_id"],
            "worker_cursor": worker_cursor,
            "final_array": state["final_array"],
        }

    # 2. If my chunk is too large, split it.
    if len(chunk) > state["max_chunk_size"]:
        midpoint = len(chunk) // 2

        left_task: AdvancedTask = {
            "task_id": state["next_task_id"],
            "start": start,
            "chunk": chunk[:midpoint],
            "operation": operation,
            "attempts": 0,
            "assigned_agent": None,
        }

        right_task: AdvancedTask = {
            "task_id": state["next_task_id"] + 1,
            "start": start + midpoint,
            "chunk": chunk[midpoint:],
            "operation": operation,
            "attempts": 0,
            "assigned_agent": None,
        }

        pending_tasks = [left_task, right_task] + pending_tasks

        agent_log.append(
            f"{agent_name} found task {task_id} too large "
            f"({len(chunk)} values), so it split it into tasks "
            f"{left_task['task_id']} and {right_task['task_id']}."
        )

        return {
            "pending_tasks": pending_tasks,
            "completed_results": completed_results,
            "agent_log": agent_log,
            "bad_task_reports": bad_task_reports,
            "next_task_id": state["next_task_id"] + 2,
            "worker_cursor": worker_cursor,
            "final_array": state["final_array"],
        }

    # 3. If I detect a bad value, report it.
    bad_values = [
        value for value in chunk
        if not isinstance(value, int)
    ]

    if bad_values:
        report = (
            f"{agent_name} detected bad values in task {task_id}: {bad_values}. "
            "It reported the issue and did not process the task."
        )

        agent_log.append(report)
        bad_task_reports.append(report)

        return {
            "pending_tasks": pending_tasks,
            "completed_results": completed_results,
            "agent_log": agent_log,
            "bad_task_reports": bad_task_reports,
            "next_task_id": state["next_task_id"],
            "worker_cursor": worker_cursor,
            "final_array": state["final_array"],
        }

    # 4. If another agent fails, take over its task.
    # We simulate one failure for selected task IDs.
    if task_id in state["simulate_failure_task_ids"] and attempts == 0:
        failed_agent = agent_name
        takeover_agent = "TakeoverPlusOneAgent"

        failed_task = dict(task)
        failed_task["attempts"] += 1
        failed_task["assigned_agent"] = takeover_agent

        pending_tasks.insert(0, failed_task)

        agent_log.append(
            f"{failed_agent} failed while processing task {task_id}. "
            f"{takeover_agent} took over the task and put it back in the queue."
        )

        return {
            "pending_tasks": pending_tasks,
            "completed_results": completed_results,
            "agent_log": agent_log,
            "bad_task_reports": bad_task_reports,
            "next_task_id": state["next_task_id"],
            "worker_cursor": worker_cursor,
            "final_array": state["final_array"],
        }

    # Normal +1 processing.
    modified_chunk = [value + 1 for value in chunk]

    # Self-validation.
    for before, after in zip(chunk, modified_chunk):
        if after != before + 1:
            raise RuntimeError(
                f"{agent_name} failed validation on task {task_id}."
            )

    completed_results.append({
        "task_id": task_id,
        "start": start,
        "modified_chunk": modified_chunk,
        "agent_name": agent_name,
    })

    agent_log.append(
        f"{agent_name} completed and validated task {task_id}, "
        f"positions {start} to {start + len(chunk) - 1}."
    )

    # 5. If I finish early, request more work.
    if pending_tasks:
        agent_log.append(
            f"{agent_name} finished early and requested more work."
        )
    else:
        agent_log.append(
            f"{agent_name} found no more pending work."
        )

    return {
        "pending_tasks": pending_tasks,
        "completed_results": completed_results,
        "agent_log": agent_log,
        "bad_task_reports": bad_task_reports,
        "next_task_id": state["next_task_id"],
        "worker_cursor": worker_cursor,
        "final_array": state["final_array"],
    }


def advanced_manager_merge_results(state: AdvancedState) -> dict:
    """
    ManagerAgent reconstructs the final array from completed chunks.
    """

    agent_log = list(state["agent_log"])

    if state["bad_task_reports"]:
        agent_log.append(
            "ManagerAgent cannot complete the final array because at least one "
            "task reported bad values."
        )
        raise RuntimeError("Bad values were reported by an agent.")

    final_array = [None] * len(state["original_array"])

    for result in state["completed_results"]:
        start = result["start"]
        chunk = result["modified_chunk"]
        final_array[start:start + len(chunk)] = chunk

    if any(value is None for value in final_array):
        agent_log.append(
            "ManagerAgent found missing positions. Some tasks were not completed."
        )
        raise RuntimeError("Final array is incomplete.")

    expected = [value + 1 for value in state["original_array"]]

    if final_array != expected:
        agent_log.append(
            "ManagerAgent validation failed: final array is incorrect."
        )
        raise RuntimeError("Final validation failed.")

    agent_log.append(
        "ManagerAgent merged all completed chunks and validated the final array."
    )

    return {
        "final_array": final_array,
        "agent_log": agent_log,
    }


def build_advanced_graph():
    graph_builder = StateGraph(AdvancedState)

    graph_builder.add_node(
        "advanced_manager_create_initial_tasks",
        advanced_manager_create_initial_tasks,
    )
    graph_builder.add_node(
        "advanced_plus_one_agent",
        advanced_plus_one_agent,
    )
    graph_builder.add_node(
        "advanced_manager_merge_results",
        advanced_manager_merge_results,
    )

    graph_builder.add_edge(START, "advanced_manager_create_initial_tasks")

    graph_builder.add_conditional_edges(
        "advanced_manager_create_initial_tasks",
        advanced_should_continue,
        {
            "advanced_plus_one_agent": "advanced_plus_one_agent",
            "advanced_manager_merge_results": "advanced_manager_merge_results",
        },
    )

    graph_builder.add_conditional_edges(
        "advanced_plus_one_agent",
        advanced_should_continue,
        {
            "advanced_plus_one_agent": "advanced_plus_one_agent",
            "advanced_manager_merge_results": "advanced_manager_merge_results",
        },
    )

    graph_builder.add_edge("advanced_manager_merge_results", END)

    return graph_builder.compile()


advanced_graph = build_advanced_graph()


def run_advanced_demo(
    original_array,
    *,
    initial_operation="+1",
    simulate_failure_task_ids=None,
    max_chunk_size=25,
    title="Advanced LangGraph demo",
):
    if simulate_failure_task_ids is None:
        simulate_failure_task_ids = []

    print(f"\n=== {title} ===\n")

    result = advanced_graph.invoke(
        {
            "original_array": original_array,
            "pending_tasks": [],
            "completed_results": [],
            "agent_log": [],
            "bad_task_reports": [],
            "next_task_id": 0,
            "max_chunk_size": max_chunk_size,
            "worker_names": [
                "PlusOneAgent-1",
                "PlusOneAgent-2",
                "PlusOneAgent-3",
                "PlusOneAgent-4",
            ],
            "worker_cursor": 0,
            "final_array": [],
            "simulate_failure_task_ids": simulate_failure_task_ids,
            "initial_operation": initial_operation,
        },
        config={"recursion_limit": 100},
    )

    print("Agent log:")
    for line in result["agent_log"]:
        print("-", line)

    print("\nFinal array:")
    print(result["final_array"])

    return result


## Run the successful Part 6 demonstration

The call deliberately introduces several situations:

```python
initial_operation="ADD_ONE_PLEASE"
simulate_failure_task_ids=[3]
max_chunk_size=25
```

This should produce a log containing:

1. clarification of the unsupported operation;
2. recursive chunk splitting;
3. a simulated first-attempt failure for task 3;
4. task requeueing and takeover;
5. repeated requests for work;
6. local validation;
7. final global validation.

The final answer should still be the array `[1, 2, ..., 100]`.


In [ ]:
successful_result = run_advanced_demo(
    list(range(100)),
    initial_operation="ADD_ONE_PLEASE",
    simulate_failure_task_ids=[3],
    max_chunk_size=25,
    title="Successful proactive LangGraph run",
)



=== Successful proactive LangGraph run ===

Agent log:
- ManagerAgent created one initial task containing the full array.
- PlusOneAgent-1 proactively requested work and received task 0.
- PlusOneAgent-1 received unsupported operation 'ADD_ONE_PLEASE' for task 0. It asked ManagerAgent for clarification.
- ManagerAgent clarified task 0: the correct operation is '+1'.
- PlusOneAgent-2 proactively requested work and received task 0.
- PlusOneAgent-2 found task 0 too large (100 values), so it split it into tasks 1 and 2.
- PlusOneAgent-3 proactively requested work and received task 1.
- PlusOneAgent-3 found task 1 too large (50 values), so it split it into tasks 3 and 4.
- PlusOneAgent-4 proactively requested work and received task 3.
- PlusOneAgent-4 failed while processing task 3. TakeoverPlusOneAgent took over the task and put it back in the queue.
- TakeoverPlusOneAgent proactively requested work and received task 3.
- TakeoverPlusOneAgent completed and validated task 3, positions 0 t

## Reading the successful Part 6 log

The log is more important than the final array because it exposes the control decisions.

Look for messages corresponding to this flow:

```text
wrong operation
      ↓
manager clarification
      ↓
100-value task split
      ↓
50-value tasks split
      ↓
one task fails on attempt 0
      ↓
task is returned to pending_tasks
      ↓
TakeoverPlusOneAgent handles it later
      ↓
all 25-value tasks complete
      ↓
manager reconstructs and validates array
```

Notice that “another agent takes over” is simulated through state and names. This notebook is teaching orchestration, not launching concurrent distributed agents in this part.


## Optional bad-value demonstration

The next cell replaces one integer with:

```python
"BAD_VALUE"
```

The worker detects the unsupported value and records a report rather than attempting arithmetic.

This demonstrates a desirable safety principle:

> A workflow should fail visibly when a required assumption is violated, rather than fabricate a complete-looking result.

The manager refuses to reconstruct the final array when `bad_task_reports` is non-empty.


In [ ]:
bad_array = list(range(100))
bad_array[7] = "BAD_VALUE"

try:
    bad_result = run_advanced_demo(
        bad_array,
        initial_operation="+1",
        simulate_failure_task_ids=[],
        max_chunk_size=25,
        title="Bad-value reporting demo",
    )
except Exception as error:
    print("\nThe graph stopped as expected.")
    print("Reason:", error)



=== Bad-value reporting demo ===


The graph stopped as expected.
Reason: Bad values were reported by an agent.


## Why the bad-value run stops

The workflow knows that successful chunks are not enough. Completeness and validity are global properties.

```text
some chunks succeeded
        +
one chunk contained invalid data
        ↓
the final array cannot be trusted
        ↓
manager raises an error
```

This is error containment:

- the invalid task is not processed;
- the problem is recorded;
- the final result is not silently produced;
- the caller receives an exception.

For a real scientific workflow, the same pattern could protect against missing files, malformed metadata, failed simulations, invalid units, or incomplete provenance.


## Part 6 takeaways: deterministic agentic orchestration

Part 6 is agentic because components:

- inspect tasks;
- select among allowed behaviours;
- update shared state;
- split and requeue work;
- recover from a simulated failure;
- continue toward a goal;
- and stop on an unsafe condition.

It is deterministic because:

- all conditions are explicit;
- the threshold is configured;
- the routes are Python functions;
- the same state follows the same branch;
- no model call is made.

```text
Part 6:
programmer defines rules
        ↓
Python evaluates conditions
        ↓
LangGraph follows routes
        ↓
Python actions update state
```

Part 7 retains this controlled graph but adds an LLM where unstructured language must be interpreted.


# Part 7: LLM-powered IoT sensor triage

The numerical `+1` scenario was ideal for explaining orchestration, but it did not contain a genuine language-understanding problem.

Part 7 uses environmental IoT readings with free-text diagnostic notes:

```python
{
    "sensor_id": "sensor-002",
    "temperature_c": 22.1,
    "battery_percent": 64,
    "diagnostic_note": "Two packets were lost after a brief network interruption..."
}
```

Some decisions remain deterministic:

```text
battery below 10%        → maintenance
temperature outside range → human review
```

Other decisions depend on the meaning of a note:

```text
temporary network interruption → probably retry
stable routine reading         → probably accept
uncertain moisture or buzzing  → human review
```

This gives the LLM a justified role: interpreting language, not performing numerical computation.


## LangGraph, LangChain, and the LLM in Part 7

| Component | Responsibility in Part 7 |
|---|---|
| **LangGraph** | Stores the batch and audit log; moves through precheck, LLM decision, action execution, and summary |
| **LangChain OpenAI integration** | Creates the `ChatOpenAI` model interface and requests structured output |
| **OpenAI LLM** | Interprets the diagnostic note and selects one permitted action |
| **Pydantic schema** | Restricts the response to an `action` and a short `reason` |
| **Python action functions** | Execute `accept`, `retry`, `maintenance`, or `human_review` |
| **Programmer-defined rules** | Handle low battery and implausible temperature before the LLM is called |

## Is this a full LangChain tool-calling agent?

Not quite. This tutorial uses a simpler bounded pattern:

```text
LLM returns a structured action name
        ↓
Python looks up the matching action function
        ↓
Python executes it
```

The action functions are ordinary Python functions, not `@tool`-decorated LangChain tools. This keeps the first LLM example transparent.

A later extension could use LangChain's higher-level agent interface and formal tool calling. The conceptual roles would remain the same.


## Add the OpenAI API key securely

The next cell asks for the key with `getpass`, so typed characters are not displayed.

```text
OPENAI_API_KEY exists?
   ├── yes → reuse it
   └── no  → ask securely and store it in the runtime environment
```

The key is stored only in the current runtime environment unless you separately save it elsewhere.

Important practical points:

- never publish an API key inside a notebook;
- never commit it to a public repository;
- API calls may incur cost;
- this demo calls the model only for readings not resolved by deterministic rules;
- model outputs can vary, so do not use this tutorial policy as a real safety policy.


In [ ]:
import getpass
import os

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass(
        "Paste your OpenAI API key: "
    )


## Part 7 architecture

```text
START
  ↓
initialise_sensor_workflow
  ↓
take_next_reading
  ↓
Is there a reading?
  ├── no  → summarise_sensor_workflow → END
  └── yes
         ↓
deterministic_precheck
         ↓
Did a hard rule decide?
  ├── yes → execute_sensor_action
  └── no  → llm_sensor_agent
                    ↓
             structured decision
                    ↓
             execute_sensor_action
                    ↓
             take_next_reading
```

The LLM appears in only one node. Everything around it remains explicit and testable.


## Reading the Part 7 definitions

### `SensorReading`

This `TypedDict` documents one input record:

- sensor identifier;
- temperature;
- battery percentage;
- free-text diagnostic note.

### `SensorDecision`

This Pydantic model defines the LLM's required output:

```python
action: Literal[
    "accept",
    "retry",
    "maintenance",
    "human_review",
]
reason: str
```

Because the action is a `Literal`, the response must use one of four names.

### `SensorState`

This is the graph's working memory:

| Field | Meaning |
|---|---|
| `input_readings` | Original sensor batch |
| `pending_readings` | Readings not yet processed |
| `current_reading` | Reading currently under inspection |
| `current_decision` | Selected action and reason |
| `results` | Completed triage records |
| `audit_log` | Chronological explanation |
| `summary` | Counts by action |


## The model and structured output

The model is created with:

```python
sensor_llm = ChatOpenAI(
    model="gpt-5-nano",
    max_retries=2,
).with_structured_output(
    SensorDecision,
    method="json_schema",
)
```

### `ChatOpenAI`

This is LangChain's OpenAI chat-model integration. It converts a Python call into a model request and converts the response back into a Python object.

### `with_structured_output`

Instead of asking for arbitrary prose, the code requests a `SensorDecision`.

This provides an important boundary:

```text
unrestricted text response
        ✗

one allowed action + short reason
        ✓
```

Structured output reduces ambiguity at the interface between probabilistic model behaviour and deterministic application code. It does not guarantee that the decision itself is factually or operationally correct, so validation and conservative policy still matter.


## Actions and policy

The program defines four ordinary Python action functions:

```text
accept()
retry()
maintenance()
human_review()
```

They are stored in a lookup table:

```python
ACTIONS = {
    "accept": accept,
    "retry": retry,
    "maintenance": maintenance,
    "human_review": human_review,
}
```

Later:

```python
action_function = ACTIONS[decision["action"]]
record = action_function(...)
```

The LLM chooses the key. Python controls what that key can execute.

### Hybrid decision policy

The workflow uses deterministic rules first:

```text
battery < 10%
    → maintenance

temperature < -40°C or > 85°C
    → human_review
```

Only unresolved cases reach the LLM.

This is a useful general pattern:

```text
hard constraints and safety rules → deterministic code
ambiguous natural-language interpretation → LLM
execution and validation → deterministic code
```


## Nodes and routers in the sensor graph

| Function | Role |
|---|---|
| `initialise_sensor_workflow()` | Copies inputs into the pending queue and starts the log |
| `take_next_reading()` | Removes one reading from the queue |
| `route_after_take()` | Chooses between processing and final summary |
| `deterministic_precheck()` | Applies hard numerical rules |
| `route_after_precheck()` | Chooses deterministic execution or the LLM |
| `llm_sensor_agent()` | Builds the prompt and requests a structured model decision |
| `execute_sensor_action()` | Calls the permitted Python function |
| `summarise_sensor_workflow()` | Counts final actions |

The prompt supplies:

- the role;
- permitted actions;
- definitions of those actions;
- conservative behavioural rules;
- and the current reading.

The LLM does not see the whole Python program. It receives the text assembled in `llm_sensor_agent()`.


In [ ]:
from collections import Counter
from typing import Literal, Optional, TypedDict

from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph
from pydantic import BaseModel, Field


class SensorReading(TypedDict):
    sensor_id: str
    temperature_c: float
    battery_percent: int
    diagnostic_note: str


class SensorDecision(BaseModel):
    action: Literal[
        "accept",
        "retry",
        "maintenance",
        "human_review",
    ] = Field(description="The single permitted action to take.")

    reason: str = Field(
        description="A concise explanation of no more than 30 words."
    )


class SensorState(TypedDict):
    input_readings: list[SensorReading]
    pending_readings: list[SensorReading]
    current_reading: Optional[SensorReading]
    current_decision: Optional[dict]
    results: list[dict]
    audit_log: list[str]
    summary: dict


# A small, inexpensive model is enough for this bounded classification task.
sensor_llm = ChatOpenAI(
    model="gpt-5-nano",
    max_retries=2,
).with_structured_output(
    SensorDecision,
    method="json_schema",
)


# ------------------------------------------------------------------
# Actions: these are ordinary Python functions defined by us.
# ------------------------------------------------------------------

def accept(reading: SensorReading, reason: str, source: str) -> dict:
    return {
        **reading,
        "action": "accept",
        "reason": reason,
        "decision_source": source,
    }


def retry(reading: SensorReading, reason: str, source: str) -> dict:
    return {
        **reading,
        "action": "retry",
        "reason": reason,
        "decision_source": source,
    }


def maintenance(reading: SensorReading, reason: str, source: str) -> dict:
    return {
        **reading,
        "action": "maintenance",
        "reason": reason,
        "decision_source": source,
    }


def human_review(reading: SensorReading, reason: str, source: str) -> dict:
    return {
        **reading,
        "action": "human_review",
        "reason": reason,
        "decision_source": source,
    }


ACTIONS = {
    "accept": accept,
    "retry": retry,
    "maintenance": maintenance,
    "human_review": human_review,
}


# ------------------------------------------------------------------
# LangGraph nodes.
# ------------------------------------------------------------------

def initialise_sensor_workflow(state: SensorState) -> dict:
    return {
        "pending_readings": list(state["input_readings"]),
        "current_reading": None,
        "current_decision": None,
        "results": [],
        "audit_log": [
            "SensorManagerAgent received the sensor batch."
        ],
        "summary": {},
    }


def take_next_reading(state: SensorState) -> dict:
    pending = list(state["pending_readings"])
    log = list(state["audit_log"])

    if not pending:
        return {
            "current_reading": None,
            "current_decision": None,
            "audit_log": log,
        }

    reading = pending.pop(0)
    log.append(
        f"SensorManagerAgent selected {reading['sensor_id']} for inspection."
    )

    return {
        "pending_readings": pending,
        "current_reading": reading,
        "current_decision": None,
        "audit_log": log,
    }


def route_after_take(state: SensorState) -> str:
    if state["current_reading"] is None:
        return "summarise"
    return "deterministic_precheck"


def deterministic_precheck(state: SensorState) -> dict:
    """
    Resolve obvious cases with explicit rules.

    These thresholds are tutorial choices supplied by the programmer.
    LangGraph and the LLM do not discover them automatically.
    """

    reading = state["current_reading"]
    assert reading is not None

    battery = reading["battery_percent"]
    temperature = reading["temperature_c"]
    log = list(state["audit_log"])

    if battery < 10:
        decision = {
            "action": "maintenance",
            "reason": (
                f"Battery is {battery}%, below the programmed "
                "10% maintenance threshold."
            ),
            "source": "deterministic rule",
        }
        log.append(
            f"A hard battery rule resolved {reading['sensor_id']}."
        )
        return {
            "current_decision": decision,
            "audit_log": log,
        }

    if temperature < -40 or temperature > 85:
        decision = {
            "action": "human_review",
            "reason": (
                f"Temperature {temperature}°C is outside the tutorial's "
                "programmed plausible range of -40°C to 85°C."
            ),
            "source": "deterministic rule",
        }
        log.append(
            f"A hard temperature rule resolved {reading['sensor_id']}."
        )
        return {
            "current_decision": decision,
            "audit_log": log,
        }

    log.append(
        f"No hard rule resolved {reading['sensor_id']}; "
        "SensorTriageAgent will interpret the diagnostic note."
    )

    return {
        "current_decision": None,
        "audit_log": log,
    }


def route_after_precheck(state: SensorState) -> str:
    if state["current_decision"] is not None:
        return "execute_action"
    return "llm_sensor_agent"


def llm_sensor_agent(state: SensorState) -> dict:
    """
    Use the LLM to interpret the unstructured diagnostic note.
    """

    reading = state["current_reading"]
    assert reading is not None

    prompt = f"""
You are SensorTriageAgent in a bounded IoT monitoring workflow.

Choose exactly one permitted action:

- accept:
  The note clearly says that the sensor is normal, stable, and trustworthy.

- retry:
  The note describes a temporary network, packet-loss, or sampling problem
  that may be fixed by taking another reading.

- maintenance:
  The note clearly describes hardware damage, calibration drift,
  battery degradation, or a physical enclosure problem.

- human_review:
  The note is ambiguous, contradictory, safety-relevant, or does not
  provide enough evidence for an automatic decision.

Rules:
- Do not invent measurements.
- Do not change the sensor values.
- Be conservative when the evidence is uncertain.
- Give a reason of no more than 30 words.

Sensor reading:
- sensor_id: {reading['sensor_id']}
- temperature_c: {reading['temperature_c']}
- battery_percent: {reading['battery_percent']}
- diagnostic_note: {reading['diagnostic_note']}
"""

    decision = sensor_llm.invoke(prompt)

    log = list(state["audit_log"])
    log.append(
        f"SensorTriageAgent selected '{decision.action}' for "
        f"{reading['sensor_id']}: {decision.reason}"
    )

    return {
        "current_decision": {
            "action": decision.action,
            "reason": decision.reason,
            "source": "LLM",
        },
        "audit_log": log,
    }


def execute_sensor_action(state: SensorState) -> dict:
    """
    Execute the selected Python action and store the result.
    """

    reading = state["current_reading"]
    decision = state["current_decision"]

    assert reading is not None
    assert decision is not None

    action_function = ACTIONS[decision["action"]]

    record = action_function(
        reading,
        decision["reason"],
        decision["source"],
    )

    results = list(state["results"])
    results.append(record)

    log = list(state["audit_log"])
    log.append(
        f"Executed action '{record['action']}' for "
        f"{reading['sensor_id']}."
    )

    return {
        "results": results,
        "audit_log": log,
    }


def summarise_sensor_workflow(state: SensorState) -> dict:
    counts = Counter(
        result["action"] for result in state["results"]
    )

    summary = {
        "accepted": counts["accept"],
        "retries": counts["retry"],
        "maintenance": counts["maintenance"],
        "human_review": counts["human_review"],
    }

    log = list(state["audit_log"])
    log.append(
        f"SensorManagerAgent completed the batch: {summary}"
    )

    return {
        "summary": summary,
        "audit_log": log,
    }


def build_sensor_graph():
    graph_builder = StateGraph(SensorState)

    graph_builder.add_node(
        "initialise_sensor_workflow",
        initialise_sensor_workflow,
    )
    graph_builder.add_node(
        "take_next_reading",
        take_next_reading,
    )
    graph_builder.add_node(
        "deterministic_precheck",
        deterministic_precheck,
    )
    graph_builder.add_node(
        "llm_sensor_agent",
        llm_sensor_agent,
    )
    graph_builder.add_node(
        "execute_sensor_action",
        execute_sensor_action,
    )
    graph_builder.add_node(
        "summarise_sensor_workflow",
        summarise_sensor_workflow,
    )

    graph_builder.add_edge(
        START,
        "initialise_sensor_workflow",
    )
    graph_builder.add_edge(
        "initialise_sensor_workflow",
        "take_next_reading",
    )

    graph_builder.add_conditional_edges(
        "take_next_reading",
        route_after_take,
        {
            "deterministic_precheck": "deterministic_precheck",
            "summarise": "summarise_sensor_workflow",
        },
    )

    graph_builder.add_conditional_edges(
        "deterministic_precheck",
        route_after_precheck,
        {
            "llm_sensor_agent": "llm_sensor_agent",
            "execute_action": "execute_sensor_action",
        },
    )

    graph_builder.add_edge(
        "llm_sensor_agent",
        "execute_sensor_action",
    )
    graph_builder.add_edge(
        "execute_sensor_action",
        "take_next_reading",
    )
    graph_builder.add_edge(
        "summarise_sensor_workflow",
        END,
    )

    return graph_builder.compile()


sensor_graph = build_sensor_graph()


def run_sensor_demo(readings: list[SensorReading]):
    result = sensor_graph.invoke(
        {
            "input_readings": readings,
            "pending_readings": [],
            "current_reading": None,
            "current_decision": None,
            "results": [],
            "audit_log": [],
            "summary": {},
        },
        config={"recursion_limit": 100},
    )

    print("Audit log:")
    for line in result["audit_log"]:
        print("-", line)

    print("\nDecisions:")
    for item in result["results"]:
        print(
            f"- {item['sensor_id']}: {item['action']} "
            f"({item['decision_source']}) — {item['reason']}"
        )

    print("\nSummary:")
    print(result["summary"])

    return result


## Run the LLM-powered sensor demonstration

The batch is designed to exercise different routes:

| Sensor | Key evidence | Expected route |
|---|---|---|
| `sensor-001` | Stable, routine, no problems | LLM → accept |
| `sensor-002` | Brief packet loss, connection returned | LLM → retry |
| `sensor-003` | Battery at 5% | deterministic rule → maintenance |
| `sensor-004` | Buzzing and uncertain moisture | LLM → human review |

The word **expected** is intentional. LLM outputs are probabilistic and may vary. The schema constrains the available actions, but it does not force one particular classification for every phrase.


In [ ]:
sensor_readings = [
    {
        "sensor_id": "sensor-001",
        "temperature_c": 21.8,
        "battery_percent": 83,
        "diagnostic_note": (
            "Routine reading. The signal is stable and no problems "
            "were observed."
        ),
    },
    {
        "sensor_id": "sensor-002",
        "temperature_c": 22.1,
        "battery_percent": 64,
        "diagnostic_note": (
            "Two packets were lost after a brief network interruption. "
            "The connection has now returned."
        ),
    },
    {
        "sensor_id": "sensor-003",
        "temperature_c": 20.9,
        "battery_percent": 5,
        "diagnostic_note": (
            "The enclosure appears dry and the measurement is plausible."
        ),
    },
    {
        "sensor_id": "sensor-004",
        "temperature_c": 19.7,
        "battery_percent": 72,
        "diagnostic_note": (
            "A technician heard intermittent buzzing and is unsure "
            "whether moisture entered the enclosure."
        ),
    },
]

sensor_result = run_sensor_demo(sensor_readings)


Audit log:
- SensorManagerAgent received the sensor batch.
- SensorManagerAgent selected sensor-001 for inspection.
- No hard rule resolved sensor-001; SensorTriageAgent will interpret the diagnostic note.
- SensorTriageAgent selected 'accept' for sensor-001: Routine reading; the signal is stable and no problems observed.
- Executed action 'accept' for sensor-001.
- SensorManagerAgent selected sensor-002 for inspection.
- No hard rule resolved sensor-002; SensorTriageAgent will interpret the diagnostic note.
- SensorTriageAgent selected 'retry' for sensor-002: Transient network interruption with brief packet loss; connection has returned. Re-reading is recommended to confirm stability.
- Executed action 'retry' for sensor-002.
- SensorManagerAgent selected sensor-003 for inspection.
- A hard battery rule resolved sensor-003.
- Executed action 'maintenance' for sensor-003.
- SensorManagerAgent selected sensor-004 for inspection.
- No hard rule resolved sensor-004; SensorTriageAgent will

## Reading the Part 7 output

The audit log should let you reconstruct the route for every reading:

```text
manager selects sensor
        ↓
hard precheck resolves it?
        ├── yes → deterministic decision source
        └── no  → SensorTriageAgent model call
                         ↓
                 LLM action and reason
        ↓
Python action executes
        ↓
manager selects next sensor
```

Every result includes `decision_source`:

```text
"deterministic rule"
or
"LLM"
```

This is important provenance. A downstream user can distinguish policy-enforced decisions from model-interpreted decisions.

The final summary counts actions but does not replace the detailed audit log.


## Why the Part 7 design is bounded

The model cannot directly:

- modify sensor values;
- change the hard thresholds;
- execute arbitrary Python;
- define a new action;
- skip the graph;
- suppress the audit log.

It can only return one schema-valid action and a short reason.

```text
LLM freedom:
choose one item from a permitted set

Application control:
define set, prompt, thresholds, execution, state, and stopping
```

This does not make the workflow automatically safe for real deployment. A production IoT system would also need:

- domain-reviewed policies;
- model evaluation;
- adversarial and failure testing;
- confidence or abstention handling;
- monitoring;
- access controls;
- human accountability;
- and careful treatment of safety-critical actions.

The tutorial illustrates architecture, not an operational sensor-safety standard.


## Part 6 versus Part 7

| Part 6 | Part 7 |
|---|---|
| Uses LangGraph | Uses LangGraph |
| All routes come from Python conditions | Some routes depend on an LLM decision |
| Inputs are structured | Includes unstructured diagnostic language |
| Thresholds and outcomes are fully programmed | Hard rules are programmed; language interpretation is delegated |
| Same state follows the same programmed branch | Model response can vary |
| No external model call | OpenAI API call for unresolved cases |
| Worker selects among `if/elif` branches | LLM selects one schema-bounded action |
| Python executes operations | Python still executes operations |

The most precise description is:

> **Part 7 is a hybrid workflow: deterministic orchestration and safety rules around a bounded LLM interpretation step.**


## Part 7 takeaways

| Question | Answer |
|---|---|
| What does the LLM do? | Interprets a diagnostic note and selects one allowed action |
| What does it not do? | Set thresholds, alter readings, or execute arbitrary code |
| What does LangChain do? | Provides the OpenAI model integration and structured-output interface |
| What does LangGraph do? | Maintains state and controls nodes, branches, and the loop |
| Who defines actions? | The programmer |
| Who executes actions? | Ordinary Python |
| Why use structured output? | To create a predictable interface between the LLM and application |
| Why keep hard rules deterministic? | They are explicit, testable, reproducible, and not dependent on language interpretation |

A useful design principle is:

```text
Use an LLM where interpretation is genuinely needed.
Do not use an LLM for rules or computations that ordinary code can express more reliably.
```


# Summary: from parallel computation to bounded agentic coordination

We began with:

```text
scatter → compute → gather
```

We then gradually added:

```text
roles
  ↓
task descriptions
  ↓
workers requesting work
  ↓
shared workflow state
  ↓
nodes and graph transitions
  ↓
inspection and validation
  ↓
loops, splitting, clarification, and recovery
  ↓
LLM interpretation of unstructured information
```

## MPI thinking and agentic-workflow thinking

| MPI-oriented view | Agentic-workflow view |
|---|---|
| Processes receive or participate in fixed work distribution | Components can request, inspect, and requeue tasks |
| Main emphasis is efficient computation and communication | Coordination and decision-making are first-class concerns |
| Control flow is encoded in the parallel program | Control flow can be represented as state, nodes, branches, and loops |
| Workers usually execute a known operation | Agents may select among bounded behaviours |
| Validation may be a final check | Validation can occur throughout the workflow |
| Failures are handled by program/runtime mechanisms | Workflow state can represent retries, takeover, reporting, and escalation |

These are complementary views. Agents do not replace MPI.

A realistic scientific architecture might look like:

```text
agentic orchestration layer
        ↓
selects, validates, retries, and explains
        ↓
workflow system / scheduler
        ↓
MPI, GPU, or other high-performance numerical kernels
```


# Final conceptual checklist

After completing the notebook, you should be able to answer these questions.

### What is an agent?

A role-bearing component that receives relevant context or state and can take actions toward a goal. It may be deterministic or LLM-powered.

### Does an agent need an LLM?

No. Parts 3–6 demonstrate deterministic agents and agentic patterns.

### What is LangGraph?

The framework that runs a stateful graph composed of Python nodes, edges, conditional routes, and loops.

### Is LangGraph deterministic?

LangGraph itself can orchestrate either deterministic or model-driven nodes.

- Parts 4–6 are deterministic because their nodes and routers are deterministic.
- Part 7 includes an LLM node, so that decision can vary.

### How does LangGraph know what to do?

It follows the nodes, edges, and router return values defined by the programmer.

### How does the workflow know that a chunk is too large?

Part 6 compares its length with the programmer-supplied `max_chunk_size`.

### What is LangChain doing?

In Part 7, it supplies the OpenAI model integration and structured-output interface.

### Where are the actions defined?

In Python code written by the programmer.

### What does the LLM add?

Interpretation of ambiguous, unstructured natural language within a bounded policy.

### What should remain deterministic?

Hard constraints, numerical calculations, validation, permission checks, and safety-critical rules whenever they can be expressed explicitly.
